In [ ]:
# default_exp paper.cnn.train_eval

# 3.2. Train & evaluate (CNN)

> Train & evaluate on multiple train/test splits with different random seeds

In [ ]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
# Python utilities
from pathlib import Path
import pickle


# mirzai utilities
from mirzai.data.loading import load_kssl
from mirzai.data.selection import (select_y, select_tax_order, select_X)
from mirzai.data.transform import log_transform_y
from mirzai.training.cnn import (Model, Evaluator, Learner)
from mirzai.data.torch import DataLoaders, SNV_transform

from fastcore.transform import compose

from sklearn.model_selection import train_test_split

# Data science stack
import numpy as np

# Deep Learning stack
import torch
from torch.optim import Adam
from torch.nn import MSELoss

import warnings
warnings.filterwarnings('ignore')

## Load and transform

In [ ]:
src_dir = '../_data'
fnames = ['spectra-features.npy', 'spectra-wavenumbers.npy', 
          'depth-order.npy', 'target.npy', 
          'tax-order-lu.pkl', 'spectra-id.npy']

X, X_names, depth_order, y, tax_lookup, X_id = load_kssl(src_dir, fnames=fnames)

data = X, y, X_id, depth_order

transforms = [select_y, select_tax_order, select_X, log_transform_y]
X, y, X_id, depth_order = compose(*transforms)(data)

## Experiment

### Setup

In [ ]:
# Is a GPU available?
use_cuda = torch.cuda.is_available()
device = torch.device('cuda:0' if use_cuda else 'cpu')
print(f'Runtime is: {device}')

Runtime is: cpu


In [ ]:
params_scheduler = {
    'base_lr': 3e-5,
    'max_lr': 1e-3,
    'step_size_up': 5,
    'mode': 'triangular',
    'cycle_momentum': False
}

n_epochs = 50
seeds = range(20)

# If no GPU then just for test
if device.type == 'cpu':
    n_epochs = 1
    seeds = range(2)
    #n_sample = 1000
    #X, y, depth_order = X[:n_sample, :], y[:n_sample], depth_order[:n_sample, :]

### Train on all Soil Taxonomic Orders and test on all and by orders

In [ ]:
evaluator = Evaluator(Model, (X, y), depth_order, 
                      seeds=seeds, device=device, n_epochs=n_epochs, verbose=False)

evaluator.train_multiple(sc_kwargs=params_scheduler)
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On all test set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=-1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=-1), '\n')

print('On all test (Mollisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=1))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=1), '\n')

print('On all test (Gelisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=12))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=12), '\n')

print('On all test (Vertisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=10))
print('Std: ', evaluator.eval_on_test(reducer=np.std, order=10), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 1.1443365, 'rpiq': 1.5732786174158195, 'r2': 0.22937549892632836, 'lccc': 0.3661954866249719, 'rmse': 1.0904665, 'mse': 1.1892555, 'mae': 0.5005693, 'mape': 82.44541883468628, 'bias': -0.07954053, 'stb': -0.15220649428847546, 'n': 36118.0}
Std:  {'rpd': 0.06289852, 'rpiq': 0.08820426295357742, 'r2': 0.08445964586855903, 'lccc': 0.05010101772183467, 'rmse': 0.011756778, 'mse': 0.025640666, 'mae': 0.024629533, 'mape': 15.134131908416748, 'bias': 0.056326114, 'stb': 0.10793319548462436, 'n': 0.0} 

On all test set
Mean:  {'rpd': 1.1342582, 'rpiq': 1.5504995234891772, 'r2': 0.2147651510777514, 'lccc': 0.363083559716014, 'rmse': 1.0849717, 'mse': 1.1783118, 'mae': 0.48764354, 'mape': 83.33835303783417, 'bias': -0.08750402, 'stb': -0.16797531183455883, 'n': 4014.0}
Std:  {'rpd': 0.06525874, 'rpiq': 0.07035731296750491, 'r2': 0.09005765251182818, 'lccc': 0.052648935738740826, 'rmse': 0.033882916, 'mse': 0.07352406, 'mae': 0.033734143, 'mape': 15.908023715019226,

### Train and test on Mollisols

In [ ]:
n_epochs = 30
seeds = range(20)

# If no GPU then just for test
if device.type == 'cpu':
    n_epochs = 1
    seeds = range(2)

In [ ]:
# Train and test on Mollisols
order = 1
mask = (depth_order[:, 1] == order) 
evaluator = Evaluator(Model, (X[mask, :], y[mask]), depth_order[mask, :], 
                      seeds=seeds, device=device, n_epochs=n_epochs, verbose=False)
evaluator.train_multiple(sc_kwargs=params_scheduler)
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Mollisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 0.70798934, 'rpiq': 0.936257181522746, 'r2': -1.0016854784769187, 'lccc': 0.04715835649468438, 'rmse': 1.1413028, 'mse': 1.3036115, 'mae': 0.7855292, 'mape': 181.9675624370575, 'bias': -0.3368059, 'stb': -0.7871022160876111, 'n': 8704.0}
Std:  {'rpd': 0.023200244, 'rpiq': 0.029464683706132755, 'r2': 0.13104654297473906, 'lccc': 0.003862654874171402, 'rmse': 0.032235682, 'mse': 0.07358134, 'mae': 0.043798983, 'mape': 11.72587275505066, 'bias': 0.02179417, 'stb': 0.05085307768796393, 'n': 0.0} 

On test (Mollisols) set
Mean:  {'rpd': 0.70748657, 'rpiq': 0.9489980436395807, 'r2': -1.0050306301605274, 'lccc': 0.04830368691885077, 'rmse': 1.0810666, 'mse': 1.1709106, 'mae': 0.7791799, 'mape': 184.5376193523407, 'bias': -0.33968124, 'stb': -0.7791798199902413, 'n': 968.0}
Std:  {'rpd': 0.020635009, 'rpiq': 0.04030291769549754, 'r2': 0.11686045288812441, 'lccc': 0.00455165237073946, 'rmse': 0.046961963, 'mse': 0.10153794, 'mae': 0.05459559, 'mape': 14.0717208385

### Train and test on Gelisols

In [ ]:
n_epochs = 10
seeds = range(20)

# If no GPU then just for test
if device.type == 'cpu':
    n_epochs = 1
    seeds = range(2)

In [ ]:
order = 12
mask = (depth_order[:, 1] == order) 
evaluator = Evaluator(Model, (X[mask, :], y[mask]), depth_order[mask, :], 
                      seeds=seeds, device=device, n_epochs=n_epochs, verbose=False)
evaluator.train_multiple(sc_kwargs=params_scheduler)
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Gelisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 0.796049, 'rpiq': 1.1420963102662958, 'r2': -0.58996658791085, 'lccc': nan, 'rmse': 1.0369645, 'mse': 1.0757216, 'mae': 0.79613423, 'mape': 245.95746994018555, 'bias': -0.3412984, 'stb': -0.5317233124327096, 'n': 358.0}
Std:  {'rpd': 0.031593323, 'rpiq': 0.048162594625534294, 'r2': 0.12600552340540871, 'lccc': nan, 'rmse': 0.020644546, 'mse': 0.042815328, 'mae': 0.022340566, 'mape': 20.825552940368652, 'bias': 0.033535257, 'stb': 0.0587522249687949, 'n': 0.0} 

On test (Gelisols) set
Mean:  {'rpd': 0.7008174, 'rpiq': 0.8375239221694697, 'r2': -1.0968863936175626, 'lccc': nan, 'rmse': 0.96625066, 'mse': 1.0199642, 'mae': 0.7102884, 'mape': 251.6355037689209, 'bias': -0.4014296, 'stb': -0.8643575520142834, 'n': 40.0}
Std:  {'rpd': 0.025968134, 'rpiq': 0.06912005476506844, 'r2': 0.15518324185537213, 'lccc': nan, 'rmse': 0.29380932, 'mse': 0.5677869, 'mae': 0.15794066, 'mape': 36.18357181549072, 'bias': 0.02078998, 'stb': 0.041950897112859764, 'n': 0.0} 



### Train and test on Vertisols

In [ ]:
n_epochs = 15
seeds = range(20)

# If no GPU then just for test
if device.type == 'cpu':
    n_epochs = 1
    seeds = range(2)

In [ ]:
order = 10
mask = (depth_order[:, 1] == order) 
evaluator = Evaluator(Model, (X[mask, :], y[mask]), depth_order[mask, :], 
                      seeds=seeds, device=device, n_epochs=n_epochs, verbose=False)
evaluator.train_multiple(sc_kwargs=params_scheduler)
print('On training set')
print('Mean: ', evaluator.eval_on_train(reducer=np.mean))
print('Std: ', evaluator.eval_on_train(reducer=np.std), '\n')

print('On test (Vertisols) set')
print('Mean: ', evaluator.eval_on_test(reducer=np.mean, order=order))
print('Std: ', evaluator.eval_on_test(reducer=np.std), '\n')

  0%|          | 0/2 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

Runtime is: cpu


  0%|          | 0/1 [00:00<?, ?it/s]

On training set
Mean:  {'rpd': 0.75864303, 'rpiq': 1.0710022078576704, 'r2': -1.004765371732237, 'lccc': nan, 'rmse': 0.74052477, 'mse': 0.5833497, 'mae': 0.63472533, 'mape': 143.92046332359314, 'bias': -0.25296646, 'stb': -0.6264010471712069, 'n': 652.0}
Std:  {'rpd': 0.16408813, 'rpiq': 0.2286400474951164, 'r2': 0.828470382992537, 'lccc': nan, 'rmse': 0.18701005, 'mse': 0.27697116, 'mae': 0.22065452, 'mape': 59.02784466743469, 'bias': 0.13235076, 'stb': 0.3285233012740739, 'n': 0.0} 

On test (Vertisols) set
Mean:  {'rpd': 0.76559937, 'rpiq': 1.1001078652583696, 'r2': -1.131820696875245, 'lccc': nan, 'rmse': 0.80205786, 'mse': 0.67185456, 'mae': 0.6813638, 'mape': 165.20682275295258, 'bias': -0.2657705, 'stb': -0.5784098432221901, 'n': 73.0}
Std:  {'rpd': 0.20076352, 'rpiq': 0.22369619705290195, 'r2': 1.0461204929303292, 'lccc': nan, 'rmse': 0.16899043, 'mse': 0.2710802, 'mae': 0.22611256, 'mape': 91.97928011417389, 'bias': 0.18251525, 'stb': 0.36617711842840467, 'n': 0.0} 



## Load trained model & predict

### Prepare data loaders/generators

In [ ]:
data = train_test_split(X, y, test_size=0.1, random_state=42)
X_train, X_test, y_train, y_test = data
dls = DataLoaders(((X_train, y_train), (X_test, y_test)), transform=SNV_transform())
training_generator, test_generator = dls.loaders()

### Load pre-trained CNN

In [ ]:
src_dir = Path('./files/models')
model = Model(X.shape[1], out_channel=16).to(device)

if device.type == 'cpu':
    model.load_state_dict(torch.load(src_dir/'model-50-epochs-02-03-2022.pt', map_location=torch.device('cpu')))
else:
    model.load_state_dict(torch.load(src_dir/'model-50-epochs-02-03-2022.pt'))

### Create Learner

Instantiate a Learner & link it to the trained model.

In [ ]:
criterion = MSELoss()
opt = Adam(model.parameters(), lr=1e-4)
learner = Learner(model, criterion, opt)
learner.model = model

Runtime is: cpu


### Predict

In [ ]:
ys_hat, y_true = learner.predict(test_generator)

In [ ]:
# Look at first 10 predicted and true values
np.c_[ys_hat, y_true]

array([[-0.6277635 , -0.66147774],
       [-0.08225732, -0.02247219],
       [-0.6254065 , -0.6400625 ],
       ...,
       [-0.4738915 , -0.64066494],
       [-0.52610576, -0.75764966],
       [-0.57999414, -0.4618278 ]], dtype=float32)

In [ ]:
# Save it if required
dest_dir = Path('./files/dumps')
with open(dest_dir/'predicted_true_cnn.pickle', 'wb') as f: 
    pickle.dump((ys_hat, y_true), f)